In [3]:
import numpy as np
import pandas as pd
from scipy.stats import poisson
import matplotlib.pyplot as plt

class XGOverperformanceAnalyzer:
    """
    Analyze whether a player's goal-scoring overperforms their xG 
    beyond random variation using Poisson test, and adjust future predictions.
    """
    
    def __init__(self, goals, xg, alpha=0.05):
        """
        Parameters:
        -----------
        goals : array-like
            Actual goals scored (per match or aggregated)
        xg : array-like
            Expected goals (xG) for same periods
        alpha : float
            Significance level for hypothesis testing (default 0.05)
        """
        self.goals = np.array(goals)
        self.xg = np.array(xg)
        self.alpha = alpha
        self.total_goals = np.sum(self.goals)
        self.total_xg = np.sum(self.xg)
        
    def poisson_test(self):
        """
        Test if total goals exceeds expected using Poisson distribution.
        
        H0: Goals follow Poisson(λ=total_xG)
        H1: Observed goals > expected (systematic overperformance)
        
        Returns:
        --------
        dict with test results including p-value and significance
        """
        # One-sided test: probability of observing this many or more goals
        # sf = survival function = P(X >= k) = 1 - cdf(k-1)
        p_value = poisson.sf(self.total_goals - 1, mu=self.total_xg)
        
        is_significant = p_value < self.alpha
        
        # Calculate confidence interval for the true rate
        # Using Poisson exact confidence interval
        observed_rate = self.total_goals / self.total_xg if self.total_xg > 0 else 0
        
        return {
            'p_value': p_value,
            'significant': is_significant,
            'goals': self.total_goals,
            'expected_goals': self.total_xg,
            'observed_rate': observed_rate,
            'alpha': self.alpha
        }
    
    def calculate_adjustment_factor(self, method='empirical_bayes', prior_weight=0.3):
        """
        Calculate adjustment factor for future xG predictions.
        
        Parameters:
        -----------
        method : str
            'simple': observed ratio (goals/xG)
            'empirical_bayes': shrink toward population mean (1.0)
            'credibility': weighted average based on sample size
        prior_weight : float
            Weight given to prior (population mean = 1.0) in empirical_bayes
            
        Returns:
        --------
        float : adjustment factor to multiply future xG by
        """
        observed_ratio = self.total_goals / self.total_xg if self.total_xg > 0 else 1.0
        
        if method == 'simple':
            return observed_ratio
        
        elif method == 'empirical_bayes':
            # Shrink toward 1.0 (no over/underperformance)
            # Higher prior_weight = more conservative (closer to 1.0)
            return prior_weight * 1.0 + (1 - prior_weight) * observed_ratio
        
        elif method == 'credibility':
            # Weight based on sample size (more data = more credibility)
            # The denominator (10) controls how much evidence is needed
            n = len(self.goals)
            credibility = n / (n + 10)
            return credibility * observed_ratio + (1 - credibility) * 1.0
        
        else:
            raise ValueError(f"Unknown method: {method}")
    
    def predict_goals(self, future_xg, adjustment_method='empirical_bayes', 
                     only_if_significant=True):
        """
        Predict future goals based on xG and overperformance adjustment.
        
        Parameters:
        -----------
        future_xg : float or array
            Expected goals for future period
        adjustment_method : str
            Method for calculating adjustment factor
            ('simple', 'empirical_bayes', 'credibility')
        only_if_significant : bool
            Only apply adjustment if overperformance is statistically significant
            
        Returns:
        --------
        dict with predicted goals, adjustment factor, and test results
        """
        # Run Poisson test
        test_result = self.poisson_test()
        
        # Calculate adjustment factor
        adjustment = self.calculate_adjustment_factor(method=adjustment_method)
        
        # Apply adjustment only if significant (optional)
        if only_if_significant and not test_result['significant']:
            adjustment = 1.0
            note = "Adjustment set to 1.0 (not statistically significant)"
        else:
            note = "Adjustment applied" if test_result['significant'] else "Adjustment applied (not significant)"
        
        predicted_goals = future_xg * adjustment
        
        return {
            'predicted_goals': predicted_goals,
            'adjustment_factor': adjustment,
            'unadjusted_prediction': future_xg,
            'test_result': test_result,
            'note': note
        }
    
    def summary_report(self):
        """Generate comprehensive analysis report."""
        print("=" * 60)
        print("xG OVERPERFORMANCE ANALYSIS REPORT (POISSON TEST)")
        print("=" * 60)
        print(f"\nData Summary:")
        print(f"  Total Goals: {self.total_goals}")
        print(f"  Total xG: {self.total_xg:.2f}")
        print(f"  Goals/xG Ratio: {self.total_goals/self.total_xg:.3f}")
        print(f"  Sample Size: {len(self.goals)} matches")
        print(f"  Difference: {self.total_goals - self.total_xg:+.2f} goals")
        
        # Run Poisson test
        test_result = self.poisson_test()
        
        print(f"\n{'Poisson Statistical Test':^60}")
        print("-" * 60)
        print(f"  Null Hypothesis: Player scores at xG rate (λ = {self.total_xg:.2f})")
        print(f"  Alternative: Player systematically overperforms")
        print(f"  P-value: {test_result['p_value']:.4f}")
        print(f"  Significance Level (α): {self.alpha}")
        
        if test_result['significant']:
            print(f"  Result: SIGNIFICANT *** (reject H0)")
            print(f"  Conclusion: Evidence of systematic overperformance")
        else:
            print(f"  Result: NOT SIGNIFICANT")
            print(f"  Conclusion: Overperformance consistent with random variation")
        
        print(f"\n{'Adjustment Factors for Future Predictions':^60}")
        print("-" * 60)
        
        methods = ['simple', 'empirical_bayes', 'credibility']
        for method in methods:
            factor = self.calculate_adjustment_factor(method=method)
            print(f"  {method:20} : {factor:.3f}")
        
        print("\n" + "=" * 60)
        
        return test_result
    
    def plot_distribution(self):
        """
        Visualize the Poisson distribution and observed goals.
        """
        fig, ax = plt.subplots(figsize=(10, 6))
        
        # Generate Poisson distribution
        x_range = np.arange(0, max(self.total_goals + 10, self.total_xg * 2))
        pmf = poisson.pmf(x_range, mu=self.total_xg)
        
        # Plot distribution
        ax.bar(x_range, pmf, alpha=0.6, label=f'Expected (Poisson λ={self.total_xg:.2f})')
        
        # Mark observed value
        ax.axvline(self.total_goals, color='red', linestyle='--', linewidth=2,
                   label=f'Observed ({self.total_goals} goals)')
        
        # Mark expected value
        ax.axvline(self.total_xg, color='blue', linestyle='--', linewidth=2,
                   label=f'Expected ({self.total_xg:.2f} xG)')
        
        test_result = self.poisson_test()
        ax.set_title(f'Poisson Test: p-value = {test_result["p_value"]:.4f}')
        ax.set_xlabel('Total Goals')
        ax.set_ylabel('Probability')
        ax.legend()
        ax.grid(alpha=0.3)
        
        plt.tight_layout()
        return fig

In [4]:

# Example Usage
if __name__ == "__main__":
    # Example 1: Player with systematic overperformance
    np.random.seed(42)
    
    print("EXAMPLE 1: Player with Systematic Overperformance")
    print("=" * 60)
    
    # Simulate a player who genuinely overperforms at 1.2x rate
    true_xg = np.random.uniform(0.1, 0.5, 100)
    goals = np.random.poisson(true_xg * 1.5) #systematic overperformance
    
    analyzer = XGOverperformanceAnalyzer(goals, true_xg, alpha=0.05)
    analyzer.summary_report()
    
    # Make future prediction
    print("\n\nFUTURE PREDICTION")
    print("=" * 60)
    future_xg_val = 1.0
    
    prediction = analyzer.predict_goals(
        future_xg=future_xg_val, 
        adjustment_method='empirical_bayes',
        only_if_significant=True
    )

    print(f"Future xG: {future_xg_val:.2f}")
    print(f"Predicted Goals (unadjusted): {prediction['unadjusted_prediction']:.2f}")
    print(f"Adjustment Factor: {prediction['adjustment_factor']:.3f}")
    print(f"Predicted Goals (adjusted): {prediction['predicted_goals']:.2f}")
    print(f"Note: {prediction['note']}")

EXAMPLE 1: Player with Systematic Overperformance
xG OVERPERFORMANCE ANALYSIS REPORT (POISSON TEST)

Data Summary:
  Total Goals: 47
  Total xG: 28.81
  Goals/xG Ratio: 1.632
  Sample Size: 100 matches
  Difference: +18.19 goals

                  Poisson Statistical Test                  
------------------------------------------------------------
  Null Hypothesis: Player scores at xG rate (λ = 28.81)
  Alternative: Player systematically overperforms
  P-value: 0.0011
  Significance Level (α): 0.05
  Result: SIGNIFICANT *** (reject H0)
  Conclusion: Evidence of systematic overperformance

         Adjustment Factors for Future Predictions          
------------------------------------------------------------
  simple               : 1.632
  empirical_bayes      : 1.442
  credibility          : 1.574



FUTURE PREDICTION
Future xG: 1.00
Predicted Goals (unadjusted): 1.00
Adjustment Factor: 1.442
Predicted Goals (adjusted): 1.44
Note: Adjustment applied
